In [1]:
import pandas as pd

# 1. Load your new Demand_data.csv
# Ensure this matches your directory structure
file_path = "inputs/Demand_data.csv"
df_demand = pd.read_csv(file_path)

# 2. Extract only the 'Time_Index' and the zonal demand columns 
# This automatically grabs Demand_MW_z1 through Demand_MW_z22
zone_cols = [col for col in df_demand.columns if col.startswith("Demand_MW_z")]
df_zones = df_demand[['Time_Index'] + zone_cols].copy()

# 3. Create an 'Hour' column (0-23) based on the Time_Index
# E.g., Time_Index 1 -> Hour 0, Time_Index 24 -> Hour 23, Time_Index 25 -> Hour 0
df_zones['Hour'] = (df_zones['Time_Index'] - 1) % 24

# 4. Calculate the average load for each zone at each hour of the day
# Group by 'Hour', drop the Time_Index, and take the mean
# This creates a DataFrame with 24 rows (hours 0-23) and 22 columns (zones)
average_diurnal_load = df_zones.drop(columns=['Time_Index']).groupby('Hour').mean()

# 5. Calculate the total system average load for each hour
total_hourly_system_load = average_diurnal_load.sum(axis=1)

# 6. Convert absolute MW to percentages
# Divide each zone's hourly average by the total system hourly average
hourly_percentages = average_diurnal_load.div(total_hourly_system_load, axis=0)

# 7. Transpose the final dataframe to match your other script's expectations
# Flips to 22 rows (zones) x 24 columns (hours)
final_output = hourly_percentages.T

# 8. Save to CSV
# We save with index=False and header=False so it's a clean matrix of numbers
final_output.to_csv("zonal_percentages.csv", index=False, header=False)

print(f"Successfully generated 24-hour zonal percentages for {len(zone_cols)} zones!")

Successfully generated 24-hour zonal percentages for 22 zones!
